In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np


from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
)

#from utils.plots import calculate_metrics, calculate_accuracy
#from utils.tools import get_config, collect_jobs, prettify_table, format_table, aggregate_results
#from utils.constants import PRED_COLS

In [3]:
POS_LABELS = ["yes", "yes", "no"]

HALLUCINATORY_LABELS = ["no", "no", "yes"]
GT_COLS = [
    "The exercise description matched the selected theme (Yes/No)",
    "The exercise description matched the selected topic (Yes/No)",
    "Included concepts that were too advanced (Yes/No)",
]

PRED_COLS = ["themeCorrect", "topicCorrect", "usesAdditionalConcepts"]

In [26]:
def normalize(series, pos_label=None):
    if pos_label is not None:
        return (
            series.astype(str)
            .str.strip('"')
            .str.lower()
            .map(lambda x: 1 if x == pos_label else 0)
        )

    try:
        pos_label = "no" if series.name == GT_COLS[2] else "yes"
    except:
        pos_label = "yes"

    return (
        series.astype(str)
        .str.strip('"')
        .str.lower()
        .map(lambda x: 1 if x == pos_label else 0)
    )

def calculate_metrics(
    df, cols1=GT_COLS, cols2=PRED_COLS, measure_hallucination_detection=True
):
    metrics = {}
    labels = ["theme", "topic", "concept"]
    for i, (c1, c2) in enumerate(zip(cols1, cols2)):
        y_true = normalize(df[c1], POS_LABELS[i])
        y_pred = normalize(df[c2], POS_LABELS[i])

        print(y_true.head(1))

        #print(y_true)
        #print(y_pred)

        # If measure_detection is True, we want to capture metrics for correctly predicted hallucinations
        # Then, the labels should be 0, 0, 1
        # Otherwise, measure correctly classified non-hallucinatory data points
        # Labels are thus 1, 1, 0
        label = 1 if HALLUCINATORY_LABELS[i] == "yes" else 0
        
        measure_label = label if measure_hallucination_detection else np.abs(label - 1)

        print(str(label) + " " +  str(measure_label))

        #print(str(label) + " " + str(measure_label))
        
        metrics[labels[i] + "_precision"] = precision_score(
            y_true, y_pred, pos_label=int(measure_label), zero_division=np.nan
        )
        metrics[labels[i] + "_recall"] = recall_score(
            y_true, y_pred, pos_label=int(measure_label), zero_division=np.nan
        )
        metrics[labels[i] + "_f1"] = f1_score(
            y_true, y_pred, pos_label=int(measure_label), zero_division=np.nan
        )  # Flip label to match labeling scheme

    return metrics

In [27]:
import pandas as pd

df = pd.read_csv("./outputs/v6/mistralai/Mistral-Small-3.2-24B-Instruct-2506/17053879/result.csv", sep=";")

metrics = calculate_metrics(df)

0    1
Name: The exercise description matched the selected theme (Yes/No), dtype: int64
0 0
0    1
Name: The exercise description matched the selected topic (Yes/No), dtype: int64
0 0
0    1
Name: Included concepts that were too advanced (Yes/No), dtype: int64
1 1


In [28]:
metrics

{'theme_precision': 0.4689922480620155,
 'theme_recall': 0.9918032786885246,
 'theme_f1': 0.6368421052631579,
 'topic_precision': 0.49429657794676807,
 'topic_recall': 1.0,
 'topic_f1': 0.6615776081424937,
 'concept_precision': 0.6715328467153284,
 'concept_recall': 0.35384615384615387,
 'concept_f1': 0.4634760705289673}